# Filtering in the Frequency Domain

**Absolute beginner → confident Fourier-domain image processing**

Master SIP · École Centrale de Nantes · Denos Kume

This notebook builds the subject progressively:

**pixels → spatial frequency → sinusoids → complex numbers → DFT → 2-D FFT → magnitude/phase → inverse FFT → LPF/HPF → Ideal/Gaussian/Butterworth → ringing → periodic noise → notch filters → moiré → shading correction → validation**

This notebook is the executable source of truth for the Filtering in the Frequency Domain laboratory. Each Required Task from 1 to 26 is presented as a numbered section with its corresponding code immediately below it. The companion notebooks contain the aligned problem definition, requirements, and theory.


## Setup — Environment and Configuration

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

np.set_printoptions(precision=3, suppress=True)

print("NumPy:", np.__version__)
print("Setup: PASS")

### 0.1 Locate the lab automatically

VS Code may execute the notebook from different working directories. We therefore search for the expected `data/Fourier` directory instead of assuming a fixed current folder.

In [ ]:
def find_lab_root():
    cwd = Path.cwd().resolve()

    for candidate in [cwd, *cwd.parents]:
        if (candidate / "data" / "Fourier").exists():
            return candidate

        nested = (
            candidate
            / "Lab_Works"
            / "Image_Processing"
            / "Filtering_in_Frequency_Domain"
        )

        if (nested / "data" / "Fourier").exists():
            return nested

    raise FileNotFoundError(
        "Could not locate Filtering_in_Frequency_Domain."
    )


LAB_ROOT = find_lab_root()
DATA_DIR = LAB_ROOT / "data"
OUTPUT_DIR = LAB_ROOT / "outputs" / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Lab root:", LAB_ROOT)
print("Data dir:", DATA_DIR)
print("Output  :", OUTPUT_DIR)

### 0.2 Reusable helpers

In [ ]:
def load_gray(path):
    return np.asarray(
        Image.open(path).convert("L"),
        dtype=np.float32
    )


def normalize01(array):
    array = np.asarray(array, dtype=np.float64)
    lo = array.min()
    hi = array.max()

    if np.isclose(lo, hi):
        return np.zeros_like(array)

    return (array - lo) / (hi - lo)


def show_gray(ax, image, title, cmap="gray"):
    ax.imshow(image, cmap=cmap)
    ax.set_title(title)
    ax.axis("off")


def save_figure(fig, filename):
    path = OUTPUT_DIR / filename
    fig.savefig(path, dpi=160, bbox_inches="tight")
    print("Saved:", path.name)


def fft2_centered(image):
    return np.fft.fftshift(np.fft.fft2(image))


def inverse_fft2_centered(F_shifted):
    return np.real(
        np.fft.ifft2(
            np.fft.ifftshift(F_shifted)
        )
    )


def log_magnitude(F_shifted):
    return np.log1p(np.abs(F_shifted))

## 1. Spatial Frequency

A sinusoidal brightness pattern can be written as:

$$
g(x)=A\sin(2\pi f x+\phi)
$$

where:

- $A$ = amplitude;
- $f$ = spatial frequency;
- $\phi$ = phase.

Low spatial frequency means intensity changes slowly across space.  
High spatial frequency means intensity changes rapidly.

**Important:** high frequency does not mean high brightness.

In [ ]:
width = 512
x = np.linspace(0, 1, width, endpoint=False)

low_signal = np.sin(2 * np.pi * 4 * x)
high_signal = np.sin(2 * np.pi * 32 * x)

low_image = np.tile(low_signal, (180, 1))
high_image = np.tile(high_signal, (180, 1))

fig, axes = plt.subplots(2, 2, figsize=(12, 6))

axes[0, 0].plot(x, low_signal)
axes[0, 0].set_title("4 cycles — low frequency")

show_gray(
    axes[0, 1],
    low_image,
    "Low spatial frequency"
)

axes[1, 0].plot(x, high_signal)
axes[1, 0].set_title("32 cycles — high frequency")

show_gray(
    axes[1, 1],
    high_image,
    "High spatial frequency"
)

fig.tight_layout()
save_figure(fig, "01_spatial_frequency.png")
plt.show()

### Interpretation

- smooth sky / gradual shading → mostly low frequency;
- fine hair / grass / texture → stronger high frequency;
- sharp edges require a combination of many frequencies.

**Checkpoint:** more cycles over the same distance means higher spatial frequency.

## 2. Sinusoids, Complex Numbers, and the DFT

The DFT of a 1-D signal is:

$$
X[k]
=
\sum_{n=0}^{N-1}
x[n]e^{-j2\pi kn/N}
$$

Inverse:

$$
x[n]
=
\frac{1}{N}
\sum_{k=0}^{N-1}
X[k]e^{j2\pi kn/N}
$$

Euler's identity:

$$
e^{j\theta}
=
\cos(\theta)+j\sin(\theta)
$$

For $X=a+jb$:

$$
|X|=\sqrt{a^2+b^2}
$$

and

$$
\phi=\operatorname{atan2}(b,a)
$$

Magnitude = frequency strength.  
Phase = spatial alignment.

In [ ]:
n = np.arange(128)

signal = (
    np.sin(2 * np.pi * 5 * n / len(n))
    + 0.45 * np.sin(2 * np.pi * 18 * n / len(n))
)

F_signal = np.fft.fft(signal)
frequencies = np.fft.fftfreq(len(signal))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(n, signal)
axes[0].set_title("Signal = two sinusoids")

axes[1].stem(
    frequencies[:64],
    np.abs(F_signal[:64])
)
axes[1].set_title("FFT magnitude")
axes[1].set_xlabel("Frequency")

fig.tight_layout()
save_figure(fig, "02_fft_1d.png")
plt.show()

## 3. The 2-D Fourier Transform for Images

For image $f(x,y)$:

$$
F(u,v)
=
\sum_{x=0}^{M-1}
\sum_{y=0}^{N-1}
f(x,y)
e^{-j2\pi\left(\frac{ux}{M}+\frac{vy}{N}\right)}
$$

The FFT computes the DFT efficiently.

`fftshift` moves the zero-frequency component to the center:

- center → low frequencies;
- farther from center → high frequencies.

Before inverse FFT, undo the shift with `ifftshift`.

In [ ]:
house = load_gray(
    DATA_DIR / "Fourier" / "house.png"
)

F_house = fft2_centered(house)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

show_gray(
    axes[0],
    house,
    "House — spatial domain"
)
show_gray(
    axes[1],
    log_magnitude(F_house),
    "Log magnitude spectrum"
)
show_gray(
    axes[2],
    np.angle(F_house),
    "Phase spectrum",
    cmap="twilight"
)

fig.tight_layout()
save_figure(fig, "03_house_spectrum.png")
plt.show()

### Why log magnitude?

Raw FFT magnitude has a very large dynamic range.

For visualization:

$$
S(u,v)=\log(1+|F(u,v)|)
$$

This makes weaker spectral components visible.

## 4. Reading a 2-D Spectrum

A useful orientation rule:

> Spatial stripes produce spectral energy perpendicular to the stripe direction.

Let's prove it visually.

In [ ]:
size = 256
coords = np.arange(size)

vertical = np.tile(
    np.sin(2 * np.pi * 16 * coords / size),
    (size, 1)
)

horizontal = vertical.T

xx, yy = np.meshgrid(coords, coords)

diagonal = np.sin(
    2 * np.pi * 12 * (xx + yy) / size
)

patterns = [
    ("Vertical stripes", vertical),
    ("Horizontal stripes", horizontal),
    ("Diagonal stripes", diagonal),
]

fig, axes = plt.subplots(3, 2, figsize=(10, 13))

for row, (name, pattern) in enumerate(patterns):
    F = fft2_centered(pattern)

    show_gray(
        axes[row, 0],
        pattern,
        name
    )
    show_gray(
        axes[row, 1],
        log_magnitude(F),
        f"{name} — spectrum"
    )

fig.tight_layout()
save_figure(fig, "04_orientation_spectra.png")
plt.show()

### Explore the supplied Fourier images

In [ ]:
fourier_files = [
    "squares.png",
    "textures.jpg",
    "tiled.png",
    "zebra-wall.png",
]

fig, axes = plt.subplots(
    len(fourier_files),
    2,
    figsize=(12, 4 * len(fourier_files))
)

for row, filename in enumerate(fourier_files):
    image = load_gray(
        DATA_DIR / "Fourier" / filename
    )

    F = fft2_centered(image)

    show_gray(
        axes[row, 0],
        image,
        filename
    )
    show_gray(
        axes[row, 1],
        log_magnitude(F),
        f"{filename} — spectrum"
    )

fig.tight_layout()
save_figure(fig, "05_dataset_spectra.png")
plt.show()

For every spectrum ask:

1. Is energy concentrated near the center?
2. Are there dominant orientations?
3. Are there isolated peaks?
4. Does the image contain repeated structure?
5. Is the spectrum approximately symmetric?

For real-valued images, conjugate symmetry often creates symmetric spectral pairs.

## 5. Inverse FFT and Reconstruction

The Fourier transform is reversible if we keep all coefficients.

In [ ]:
reconstructed_house = inverse_fft2_centered(
    F_house
)

absolute_error = np.abs(
    house.astype(np.float64)
    - reconstructed_house
)

print(
    "Maximum reconstruction error:",
    f"{absolute_error.max():.6e}"
)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

show_gray(
    axes[0],
    house,
    "Original"
)
show_gray(
    axes[1],
    reconstructed_house,
    "IFFT reconstruction"
)
show_gray(
    axes[2],
    absolute_error,
    "Absolute error"
)

fig.tight_layout()
save_figure(fig, "06_reconstruction.png")
plt.show()

Tiny imaginary terms after IFFT are numerical roundoff when reconstructing an originally real image. Taking the real part is appropriate when those imaginary values are negligible.

## 6. Magnitude vs Phase

Every Fourier coefficient can be written as:

$$
F(u,v)=|F(u,v)|e^{j\phi(u,v)}
$$

Magnitude tells us how strong a frequency is.  
Phase strongly controls spatial organization.

In [ ]:
cat = load_gray(
    DATA_DIR / "PhaseMag" / "cat.jpg"
)

wolf = load_gray(
    DATA_DIR / "PhaseMag" / "wolf.jpg"
)

common_size = (256, 256)

cat_r = np.asarray(
    Image.fromarray(
        cat.astype(np.uint8)
    ).resize(common_size),
    dtype=np.float32
)

wolf_r = np.asarray(
    Image.fromarray(
        wolf.astype(np.uint8)
    ).resize(common_size),
    dtype=np.float32
)

F_cat = np.fft.fft2(cat_r)
F_wolf = np.fft.fft2(wolf_r)

mag_cat = np.abs(F_cat)
phase_cat = np.angle(F_cat)

mag_wolf = np.abs(F_wolf)
phase_wolf = np.angle(F_wolf)

cat_mag_wolf_phase = np.real(
    np.fft.ifft2(
        mag_cat
        * np.exp(1j * phase_wolf)
    )
)

wolf_mag_cat_phase = np.real(
    np.fft.ifft2(
        mag_wolf
        * np.exp(1j * phase_cat)
    )
)

fig, axes = plt.subplots(2, 2, figsize=(10, 10))

show_gray(
    axes[0, 0],
    cat_r,
    "Original cat"
)
show_gray(
    axes[0, 1],
    wolf_r,
    "Original wolf"
)
show_gray(
    axes[1, 0],
    cat_mag_wolf_phase,
    "Cat magnitude + wolf phase"
)
show_gray(
    axes[1, 1],
    wolf_mag_cat_phase,
    "Wolf magnitude + cat phase"
)

fig.tight_layout()
save_figure(fig, "07_phase_magnitude_swap.png")
plt.show()

A strong interview-level summary:

> Magnitude describes the strength of frequencies; phase preserves much of the spatial organization of structures.

In [ ]:
magnitude_only = np.real(
    np.fft.ifft2(
        mag_cat
        * np.exp(
            1j * np.zeros_like(phase_cat)
        )
    )
)

phase_only = np.real(
    np.fft.ifft2(
        np.ones_like(mag_cat)
        * np.exp(1j * phase_cat)
    )
)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

show_gray(
    axes[0],
    cat_r,
    "Original"
)
show_gray(
    axes[1],
    magnitude_only,
    "Magnitude only"
)
show_gray(
    axes[2],
    phase_only,
    "Phase only"
)

fig.tight_layout()
save_figure(
    fig,
    "08_phase_only_magnitude_only.png"
)
plt.show()

## 7. Frequency-Domain Filtering

Let $F$ be the image spectrum and $H$ the filter:

$$
G(u,v)=H(u,v)F(u,v)
$$

Then:

$$
g(x,y)=\mathcal{F}^{-1}\{G(u,v)\}
$$

Workflow:

1. FFT;
2. center with `fftshift`;
3. construct $H$;
4. multiply $H\cdot F$;
5. undo shift;
6. IFFT;
7. keep the real component.

## 8. Frequency Distance Grid

For circular filters:

$$
D(u,v)=
\sqrt{(u-u_0)^2+(v-v_0)^2}
$$

In [ ]:
def frequency_distance_grid(shape):
    rows, cols = shape
    cy = rows // 2
    cx = cols // 2

    y, x = np.ogrid[:rows, :cols]

    return np.sqrt(
        (y - cy) ** 2
        + (x - cx) ** 2
    )


D_house = frequency_distance_grid(
    house.shape
)

plt.figure(figsize=(6, 5))
plt.imshow(
    D_house,
    cmap="viridis"
)
plt.title("Distance from frequency origin")
plt.colorbar(
    label="Frequency-bin distance"
)
plt.axis("off")
plt.show()

## 9. Ideal, Gaussian, and Butterworth Low-Pass Filters

### Ideal LPF

$$
H(u,v)=
\begin{cases}
1,&D(u,v)\le D_0\\
0,&D(u,v)>D_0
\end{cases}
$$

### Gaussian LPF

$$
H(u,v)
=
\exp\left(
-\frac{D(u,v)^2}{2D_0^2}
\right)
$$

### Butterworth LPF

$$
H(u,v)
=
\frac{1}
{1+\left(\frac{D(u,v)}{D_0}\right)^{2n}}
$$

Butterworth order $n$ controls transition steepness.

In [ ]:
def ideal_low_pass(shape, cutoff):
    D = frequency_distance_grid(shape)
    return (D <= cutoff).astype(np.float32)


def gaussian_low_pass(shape, cutoff):
    D = frequency_distance_grid(shape)

    return np.exp(
        -(D ** 2)
        / (2 * cutoff ** 2)
    )


def butterworth_low_pass(
    shape,
    cutoff,
    order=2
):
    D = frequency_distance_grid(shape)

    return 1.0 / (
        1.0
        + (
            D
            / max(float(cutoff), 1e-12)
        ) ** (2 * order)
    )


def apply_frequency_filter(
    image,
    H
):
    F = fft2_centered(image)
    G = F * H
    result = inverse_fft2_centered(G)

    return result, F, G


cutoff = 30

H_ideal_lp = ideal_low_pass(
    house.shape,
    cutoff
)

H_gaussian_lp = gaussian_low_pass(
    house.shape,
    cutoff
)

H_butterworth_lp = butterworth_low_pass(
    house.shape,
    cutoff,
    order=2
)

house_ideal_lp, _, _ = apply_frequency_filter(
    house,
    H_ideal_lp
)

house_gaussian_lp, _, _ = apply_frequency_filter(
    house,
    H_gaussian_lp
)

house_butterworth_lp, _, _ = apply_frequency_filter(
    house,
    H_butterworth_lp
)

fig, axes = plt.subplots(
    2,
    3,
    figsize=(15, 10)
)

show_gray(
    axes[0, 0],
    H_ideal_lp,
    "Ideal LPF"
)

show_gray(
    axes[0, 1],
    H_gaussian_lp,
    "Gaussian LPF"
)

show_gray(
    axes[0, 2],
    H_butterworth_lp,
    "Butterworth LPF"
)

show_gray(
    axes[1, 0],
    house_ideal_lp,
    "Ideal result"
)

show_gray(
    axes[1, 1],
    house_gaussian_lp,
    "Gaussian result"
)

show_gray(
    axes[1, 2],
    house_butterworth_lp,
    "Butterworth result"
)

fig.tight_layout()
save_figure(
    fig,
    "09_lpf_comparison.png"
)
plt.show()

### Comparison

| Filter | Transition | Ringing tendency |
|---|---|---|
| Ideal | abrupt | high |
| Gaussian | very smooth | very low |
| Butterworth | adjustable | depends on order |

There is no universal best filter.

In [ ]:
orders = [1, 2, 5]

fig, axes = plt.subplots(
    len(orders),
    2,
    figsize=(11, 12)
)

for row, order in enumerate(orders):
    H = butterworth_low_pass(
        house.shape,
        cutoff,
        order=order
    )

    result, _, _ = apply_frequency_filter(
        house,
        H
    )

    show_gray(
        axes[row, 0],
        H,
        f"Butterworth n={order}"
    )

    show_gray(
        axes[row, 1],
        result,
        f"Result n={order}"
    )

fig.tight_layout()
save_figure(
    fig,
    "10_butterworth_orders.png"
)
plt.show()

Higher Butterworth order makes the transition sharper. Sharper transitions can also increase ringing tendency.

## 10. Ringing and the Gibbs Phenomenon

A hard spectral cutoff corresponds to an oscillatory spatial response:

> abrupt spectral boundary → spatial oscillations → halos near edges

In [ ]:
square = np.zeros(
    (256, 256),
    dtype=np.float32
)

square[64:192, 64:192] = 255.0

ring_cutoff = 22

Hi = ideal_low_pass(
    square.shape,
    ring_cutoff
)

Hg = gaussian_low_pass(
    square.shape,
    ring_cutoff
)

Hb = butterworth_low_pass(
    square.shape,
    ring_cutoff,
    order=2
)

square_i, _, _ = apply_frequency_filter(
    square,
    Hi
)

square_g, _, _ = apply_frequency_filter(
    square,
    Hg
)

square_b, _, _ = apply_frequency_filter(
    square,
    Hb
)

center_row = square.shape[0] // 2

fig, axes = plt.subplots(
    2,
    2,
    figsize=(13, 10)
)

show_gray(
    axes[0, 0],
    square,
    "Original square"
)

show_gray(
    axes[0, 1],
    square_i,
    "Ideal LPF — ringing"
)

axes[1, 0].plot(
    square[center_row],
    label="Original"
)

axes[1, 0].plot(
    square_i[center_row],
    label="Ideal"
)

axes[1, 0].plot(
    square_g[center_row],
    label="Gaussian"
)

axes[1, 0].plot(
    square_b[center_row],
    label="Butterworth"
)

axes[1, 0].set_title(
    "Intensity profile through edge"
)
axes[1, 0].legend()

show_gray(
    axes[1, 1],
    np.abs(square_i - square),
    "Ideal LPF difference"
)

fig.tight_layout()
save_figure(
    fig,
    "11_ringing.png"
)
plt.show()

## 11. High-Pass Filtering

For a normalized LPF:

$$
H_{HP}=1-H_{LP}
$$

High frequencies contain edges and fine detail, but can also contain noise.

In [ ]:
def high_pass_from_low_pass(H_low):
    return 1.0 - H_low


H_ideal_hp = high_pass_from_low_pass(
    H_ideal_lp
)

H_gaussian_hp = high_pass_from_low_pass(
    H_gaussian_lp
)

H_butterworth_hp = high_pass_from_low_pass(
    H_butterworth_lp
)

house_i_hp, _, _ = apply_frequency_filter(
    house,
    H_ideal_hp
)

house_g_hp, _, _ = apply_frequency_filter(
    house,
    H_gaussian_hp
)

house_b_hp, _, _ = apply_frequency_filter(
    house,
    H_butterworth_hp
)

fig, axes = plt.subplots(
    2,
    3,
    figsize=(15, 10)
)

show_gray(
    axes[0, 0],
    H_ideal_hp,
    "Ideal HPF"
)

show_gray(
    axes[0, 1],
    H_gaussian_hp,
    "Gaussian HPF"
)

show_gray(
    axes[0, 2],
    H_butterworth_hp,
    "Butterworth HPF"
)

show_gray(
    axes[1, 0],
    normalize01(house_i_hp),
    "Ideal HP response"
)

show_gray(
    axes[1, 1],
    normalize01(house_g_hp),
    "Gaussian HP response"
)

show_gray(
    axes[1, 2],
    normalize01(house_b_hp),
    "Butterworth HP response"
)

fig.tight_layout()
save_figure(
    fig,
    "12_high_pass.png"
)
plt.show()

## 12. High-Boost Sharpening

A pure high-pass result mainly contains detail.

For sharpening:

$$
g(x,y)=f(x,y)+k f_{HP}(x,y)
$$

In [ ]:
k = 1.2

high_detail = house_g_hp

high_boost = np.clip(
    house + k * high_detail,
    0,
    255
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

show_gray(
    axes[0],
    house,
    "Original"
)

show_gray(
    axes[1],
    normalize01(high_detail),
    "High-frequency detail"
)

show_gray(
    axes[2],
    high_boost,
    f"High-boost result — k={k}"
)

fig.tight_layout()
save_figure(
    fig,
    "13_high_boost.png"
)
plt.show()

## 13. Convolution Theorem

$$
f*h
\quad\Longleftrightarrow\quad
F\cdot H
$$

Spatial convolution corresponds to multiplication in the frequency domain.

### Circular vs Linear Convolution

A DFT assumes periodic extension.

Therefore direct FFT multiplication naturally performs **circular convolution**.  
For ordinary linear convolution, appropriate zero-padding is generally required.

## 14. Band-Pass and Band-Reject Filters

Band-pass keeps:

$$
D_1\le D(u,v)\le D_2
$$

Band-reject removes that interval.

In [ ]:
def ideal_band_pass(
    shape,
    low_cutoff,
    high_cutoff
):
    D = frequency_distance_grid(shape)

    return (
        (D >= low_cutoff)
        & (D <= high_cutoff)
    ).astype(np.float32)


H_band = ideal_band_pass(
    house.shape,
    15,
    55
)

H_band_reject = 1.0 - H_band

house_band, _, _ = apply_frequency_filter(
    house,
    H_band
)

house_band_reject, _, _ = apply_frequency_filter(
    house,
    H_band_reject
)

fig, axes = plt.subplots(
    2,
    2,
    figsize=(11, 10)
)

show_gray(
    axes[0, 0],
    H_band,
    "Band-pass mask"
)

show_gray(
    axes[0, 1],
    normalize01(house_band),
    "Band-pass content"
)

show_gray(
    axes[1, 0],
    H_band_reject,
    "Band-reject mask"
)

show_gray(
    axes[1, 1],
    house_band_reject,
    "Band-reject result"
)

fig.tight_layout()
save_figure(
    fig,
    "14_band_filters.png"
)
plt.show()

## 15. Periodic Noise

Periodic interference is one of the strongest reasons to use the frequency domain.

Repeated interference often becomes isolated off-center peaks in the spectrum.

In [ ]:
interference_files = [
    "astronaut-interference.tif",
    "car-moire-pattern.tif",
]

fig, axes = plt.subplots(
    2,
    2,
    figsize=(12, 10)
)

for row, filename in enumerate(interference_files):
    image = load_gray(
        DATA_DIR
        / "Frequency"
        / filename
    )

    F = fft2_centered(image)

    show_gray(
        axes[row, 0],
        image,
        filename
    )

    show_gray(
        axes[row, 1],
        log_magnitude(F),
        f"{filename} — spectrum"
    )

fig.tight_layout()
save_figure(
    fig,
    "15_periodic_noise_spectra.png"
)
plt.show()

Look for:

- isolated off-center peaks;
- symmetric peak pairs;
- spectral lines;
- patterns clearly associated with repetition.

A bright peak is not automatically noise. Useful repetitive texture can also create peaks.

## 16. Spectral Peak Detection

The following detector is intentionally simple:

1. remove the central low-frequency area;
2. rank remaining coefficients;
3. keep strong points separated by a minimum distance.

In [ ]:
def strongest_spectral_peaks(
    F_shifted,
    number_of_peaks=8,
    center_exclusion_radius=20,
    min_separation=10
):
    magnitude = log_magnitude(
        F_shifted
    ).copy()

    rows, cols = magnitude.shape
    cy, cx = rows // 2, cols // 2

    yy, xx = np.ogrid[:rows, :cols]

    center_mask = (
        (yy - cy) ** 2
        + (xx - cx) ** 2
        <= center_exclusion_radius ** 2
    )

    magnitude[center_mask] = -np.inf

    flat_order = np.argsort(
        magnitude.ravel()
    )[::-1]

    selected = []

    for flat_index in flat_order:
        y, x = np.unravel_index(
            flat_index,
            magnitude.shape
        )

        separated = all(
            (y - py) ** 2
            + (x - px) ** 2
            >= min_separation ** 2
            for py, px in selected
        )

        if separated:
            selected.append((y, x))

        if len(selected) >= number_of_peaks:
            break

    return selected

## 17. Notch-Reject Filtering

A notch-reject filter suppresses a small neighborhood around selected unwanted frequencies.

Real images have conjugate-symmetric spectra, so corresponding symmetric frequencies must also be considered.

In [ ]:
def notch_reject_mask(
    shape,
    peak_locations,
    radius=5
):
    rows, cols = shape
    cy, cx = rows // 2, cols // 2

    yy, xx = np.ogrid[:rows, :cols]

    mask = np.ones(
        shape,
        dtype=np.float32
    )

    for py, px in peak_locations:
        d1 = (
            (yy - py) ** 2
            + (xx - px) ** 2
        )

        mask[d1 <= radius ** 2] = 0.0

        sym_y = 2 * cy - py
        sym_x = 2 * cx - px

        d2 = (
            (yy - sym_y) ** 2
            + (xx - sym_x) ** 2
        )

        mask[d2 <= radius ** 2] = 0.0

    return mask

In [ ]:
astronaut = load_gray(
    DATA_DIR
    / "Frequency"
    / "astronaut-interference.tif"
)

F_astronaut = fft2_centered(
    astronaut
)

astronaut_peaks = strongest_spectral_peaks(
    F_astronaut,
    number_of_peaks=8,
    center_exclusion_radius=25,
    min_separation=12
)

astronaut_notch = notch_reject_mask(
    astronaut.shape,
    astronaut_peaks[:4],
    radius=4
)

astronaut_filtered = inverse_fft2_centered(
    F_astronaut
    * astronaut_notch
)

fig, axes = plt.subplots(
    2,
    2,
    figsize=(12, 10)
)

show_gray(
    axes[0, 0],
    astronaut,
    "Original interference"
)

show_gray(
    axes[0, 1],
    log_magnitude(F_astronaut),
    "Original spectrum"
)

show_gray(
    axes[1, 0],
    astronaut_notch,
    "Notch-reject mask"
)

show_gray(
    axes[1, 1],
    astronaut_filtered,
    "After notch filtering"
)

fig.tight_layout()
save_figure(
    fig,
    "16_notch_filter.png"
)
plt.show()

print(
    "Candidate peaks:",
    astronaut_peaks
)

### Notch-radius trade-off

- too small → interference remains;
- too large → useful nearby frequencies are removed.

The goal is selective suppression.

## 18. Moiré Removal

Moiré is a repeated interference pattern. It can often be easier to isolate in the Fourier domain than in the spatial domain.

In [ ]:
car_moire = load_gray(
    DATA_DIR
    / "Frequency"
    / "car-moire-pattern.tif"
)

F_car = fft2_centered(
    car_moire
)

car_peaks = strongest_spectral_peaks(
    F_car,
    number_of_peaks=10,
    center_exclusion_radius=30,
    min_separation=12
)

car_notch = notch_reject_mask(
    car_moire.shape,
    car_peaks[:6],
    radius=4
)

car_filtered = inverse_fft2_centered(
    F_car * car_notch
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(16, 5)
)

show_gray(
    axes[0],
    car_moire,
    "Car with moiré"
)

show_gray(
    axes[1],
    log_magnitude(F_car),
    "Moiré spectrum"
)

show_gray(
    axes[2],
    car_filtered,
    "Notch-filtered result"
)

fig.tight_layout()
save_figure(
    fig,
    "17_moire_removal.png"
)
plt.show()

## 19. Slowly Varying Illumination / Shading

A simple multiplicative model is:

$$
I(x,y)\approx R(x,y)L(x,y)
$$

where:

- $R$ = reflectance / useful structure;
- $L$ = slowly varying illumination.

Because illumination varies slowly, it is dominated by low frequencies.

In [ ]:
spotshade = load_gray(
    DATA_DIR
    / "Frequency"
    / "text-spotshade.tif"
)

illumination_filter = gaussian_low_pass(
    spotshade.shape,
    cutoff=18
)

illumination, _, _ = apply_frequency_filter(
    spotshade,
    illumination_filter
)

epsilon = 1e-6

corrected = spotshade / (
    illumination + epsilon
)

corrected = normalize01(
    corrected
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

show_gray(
    axes[0],
    spotshade,
    "Original shaded image"
)

show_gray(
    axes[1],
    illumination,
    "Estimated illumination"
)

show_gray(
    axes[2],
    corrected,
    "Normalized image"
)

fig.tight_layout()
save_figure(
    fig,
    "18_shading_correction.png"
)
plt.show()

For additive background variation, subtraction can be more appropriate.

The correction strategy should match the assumed image-formation model.

## 20. Cutoff Sensitivity

For a low-pass filter:

- smaller cutoff → stronger smoothing;
- larger cutoff → more detail preserved.

In [ ]:
cutoffs = [10, 25, 60, 120]

fig, axes = plt.subplots(
    2,
    len(cutoffs),
    figsize=(4 * len(cutoffs), 8)
)

for col, current_cutoff in enumerate(cutoffs):
    H = gaussian_low_pass(
        house.shape,
        current_cutoff
    )

    filtered, _, _ = apply_frequency_filter(
        house,
        H
    )

    show_gray(
        axes[0, col],
        H,
        f"Gaussian LPF\nD0={current_cutoff}"
    )

    show_gray(
        axes[1, col],
        filtered,
        f"Result\nD0={current_cutoff}"
    )

fig.tight_layout()
save_figure(
    fig,
    "19_cutoff_sensitivity.png"
)
plt.show()

## 21. Quantitative Checks

MSE:

$$
\mathrm{MSE}
=
\frac{1}{MN}
\sum_{x,y}
[f(x,y)-g(x,y)]^2
$$

PSNR:

$$
\mathrm{PSNR}
=
10\log_{10}
\left(
\frac{255^2}{\mathrm{MSE}}
\right)
$$

PSNR measures numerical fidelity to a reference. It is not a universal perceptual-quality metric.

In [ ]:
def mse(reference, test):
    reference = np.asarray(
        reference,
        dtype=np.float64
    )

    test = np.asarray(
        test,
        dtype=np.float64
    )

    return np.mean(
        (reference - test) ** 2
    )


def psnr(reference, test, peak=255.0):
    error = mse(reference, test)

    if np.isclose(error, 0.0):
        return np.inf

    return 10 * np.log10(
        (peak ** 2) / error
    )


methods = {
    "Ideal LPF": house_ideal_lp,
    "Gaussian LPF": house_gaussian_lp,
    "Butterworth LPF": house_butterworth_lp,
}

for name, result in methods.items():
    print(
        f"{name:18s} "
        f"MSE={mse(house, result):10.3f} "
        f"PSNR={psnr(house, result):7.2f} dB"
    )

Here the house image is not a noisy observation with a known clean reference.

So these values mainly quantify how much each filter changes the original image.

## 22. Validation Checks

In [ ]:
assert reconstructed_house.shape == house.shape
assert absolute_error.max() < 1e-8

filters_to_check = [
    H_ideal_lp,
    H_gaussian_lp,
    H_butterworth_lp,
    H_ideal_hp,
    H_gaussian_hp,
    H_butterworth_hp,
]

for H in filters_to_check:
    assert H.shape == house.shape
    assert np.isfinite(H).all()
    assert H.min() >= 0
    assert H.max() <= 1

center = (
    house.shape[0] // 2,
    house.shape[1] // 2
)

assert np.isclose(
    H_ideal_lp[center],
    1.0
)

assert np.isclose(
    H_gaussian_lp[center],
    1.0
)

assert np.isclose(
    H_butterworth_lp[center],
    1.0
)

assert np.isclose(
    H_ideal_hp[center],
    0.0
)

assert np.isclose(
    H_gaussian_hp[center],
    0.0
)

assert np.isclose(
    H_butterworth_hp[center],
    0.0
)

print(
    "PASS — reconstruction and filter "
    "sanity checks"
)

## 23. Common Mistakes

1. Displaying raw FFT magnitude instead of `log1p(abs(F))`.
2. Forgetting `fftshift`.
3. Forgetting `ifftshift`.
4. Confusing brightness with frequency.
5. Assuming all high frequencies are noise.
6. Using Ideal filters without expecting ringing.
7. Removing bright peaks blindly.
8. Ignoring circular convolution and zero-padding.
9. Treating PSNR as universal perceptual quality.
10. Memorizing formulas without linking spectrum to spatial structure.

## 24. Practical Exercises

### Beginner

1. Generate 4, 8, 16, and 32-cycle gratings.
2. Predict FFT peak locations before running.
3. Compare vertical, horizontal, and diagonal stripes.
4. Verify FFT → IFFT reconstruction.

### Intermediate

5. Test $D_0\in\{10,20,40,80\}$.
6. Test Butterworth $n\in\{1,2,4,8\}$.
7. Compare ringing on a binary square.
8. Repeat magnitude/phase swapping with another image pair.
9. Explain when HPF also amplifies noise.

### Advanced

10. Choose notch locations manually for `astronaut-interference.tif`.
11. Compare manual notches with automatic peak candidates.
12. Tune notch radius.
13. Remove moiré from `car-moire-pattern.tif`.
14. Tune shading correction on `text-spotshade.tif`.
15. Add zero-padding and demonstrate circular vs linear convolution.

## 25. Interview / Exam Questions

**What does the Fourier transform represent?**  
2-D spatial-frequency content as complex coefficients.

**Why use `fftshift`?**  
To center zero frequency for interpretation and filter design.

**Why use log magnitude?**  
FFT magnitude has a very large dynamic range.

**Magnitude vs phase?**  
Magnitude gives component strength; phase carries spatial alignment and much structural information.

**Core filtering equation?**

$$
G(u,v)=H(u,v)F(u,v)
$$

**Why does Ideal LPF ring?**  
Its abrupt cutoff corresponds to an oscillatory spatial response.

**What does Butterworth order control?**  
Transition steepness.

**Why is periodic noise often easier in Fourier domain?**  
It can appear as localized spectral peaks.

**Why symmetric peak pairs?**  
Real-valued images have conjugate-symmetric Fourier spectra.

**Convolution theorem?**  
Spatial convolution corresponds to frequency multiplication.

**Why can FFT filtering wrap around?**  
The DFT assumes periodic extension and naturally gives circular convolution.

## 26. Final Concept Map

```text
SPATIAL IMAGE
     │
     ├── smooth variation ─────────► low frequencies
     ├── edges / detail ───────────► high frequencies
     └── periodic patterns ────────► spectral peaks
                         │
                         ▼
                       FFT2
                         │
                      fftshift
                         │
             ┌───────────┴───────────┐
             │                       │
         MAGNITUDE                 PHASE
       frequency strength      spatial organization
             │                       │
             └───────────┬───────────┘
                         │
                    H(u,v) × F
                         │
                     ifftshift
                         │
                       IFFT2
                         │
                         ▼
                  FILTERED IMAGE
```

## Final Result Summary

The notebook implements and validates the complete frequency-domain workflow, from the 2-D Fourier transform and spectrum interpretation through low/high/band/notch filtering, periodic-noise removal, illumination analysis, and quantitative checks. Generated figures are stored under `../outputs/figures/`.
